# **SWAG & AF**


*   INPUT: CSV after preprocess (right format)
*   OUTPUT: CSV after Adversial Filtering.

**Hyper Parameters:**

*   INPUT: CSV after preprocess (right format)
*   OUTPUT: CSV after Adversial Filtering.
*   FRACTION: sample a fraction of the benchmark (for example 0.1 for 10%)
*   all AF - related HyperParameters


**LOGS & METRICS**

* Graphs of conversion
* Per iteration per epoch logs
* Per source precentage remaining
* Per Category precetage remaining


In [ ]:
# Hyper Parameters — everything tunable lives here

# --- INPUT / OUTPUT ---
DRIVE_ROOT  = '/content/drive/MyDrive/@@@@@@@@@@'
DATA_PATH   = 'ANON'
INPUT_CSV_PATH  = f'{DRIVE_ROOT}/{DATA_PATH}/@@@@@@@@@@@@@@@@.csv'    # expansion benchmark (Scripts 2c product, 4.37M rows + metadata)
OUTPUT_CSV_PATH = f'{DRIVE_ROOT}/{DATA_PATH}/@@@@@@@@@@@@@@@.csv' # AF-filtered benchmark (this notebook's product; keeps all metadata cols)


# --- Sampling ---
FRACTION = 1.0    # per-source fraction of the input to use (e.g. 0.1 = 10%); 1.0 = all

# --- SWAG training ---
GLOVE_DIM   = 100
NUM_EPOCHS  = 10
BATCH_SIZE  = 512
LR          = 1e-3
TOP_K_WORDS = 100

# --- AF main loop ---
REMOVAL_RATE    = 0.10   # fraction of easy pairs removed per iteration
EASY_THRESHOLD  = 0.5    # ensemble score above this = "easy"
ACCURACY_TARGET = 0.55   # stop when ensemble val accuracy <= this
MIN_RETENTION   = 0.2   # stop when benchmark shrinks below this
MAX_ITERS       = 120

# --- Cleanup phase (reverse artifacts) ---
CLEANUP_THRESHOLD    = 0.47
CLEANUP_REMOVAL_RATE = 0.10
CLEANUP_MAX_ITERS    = 20
CLEANUP_MAX_REMOVAL  = 0.05
CLEANUP_TARGET       = 0.47

# --- Checkpoint / resume (for multi-session runs on big benchmarks) ---
# Every AF run gets its OWN subfolder under CKPT_ROOT (auto-named run_YYYYMMDD_HHMM).
# Resume finds the latest UNFINISHED run; finished runs are never touched.
CKPT_ROOT        = f'{DRIVE_ROOT}/{DATA_PATH}/af_ckpt'
RUN_NAME         = None     # None = auto (resume latest unfinished, else new run folder);
                            # or pin explicitly, e.g. 'run_2600k_b512'
RESUME           = True     # True: auto-resume the latest unfinished run in CKPT_ROOT
CHECKPOINT_EVERY = 2        # save state every N AF iterations

MAX_TRAIN_PAIRS_PER_ITER = None   
                                  
SPACY_PROCS = 2                   # spaCy tokenization processes (one-time cost per session)

SEED = 42
print("Hyperparameters set.")


Hyperparameters set.


# **Section 1: Utilities**

Setup that everything else depends on: pip installs, Drive mount, path configuration, raw data loading.

Run this once at the start of every session.


In [ ]:
# Section 1.1 — Pip installs (run once per Colab session)

!pip install -q spacy
!python -m spacy download en_core_web_sm -q

print("Dependencies installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 216.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Dependencies installed.


In [ ]:
# Section 1.2 — Imports, Drive mount, seeding

import os, sys, json, pickle, re, zipfile, urllib.request
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

from google.colab import drive
drive.mount('/content/drive')

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"INPUT_CSV_PATH  = {INPUT_CSV_PATH}")
print(f"OUTPUT_CSV_PATH = {OUTPUT_CSV_PATH}")
print(f"FRACTION        = {FRACTION}")


In [ ]:
# Section 1.3 — Load the preprocessed benchmark (+ optional per-source FRACTION sample)
#
# INPUT is the already-expanded (source, close, far[, group]) CSV produced by 04_Scripts.
# The raw-CSV expansion itself lives in the Scripts notebook, not here.

assert os.path.exists(INPUT_CSV_PATH), f"input benchmark not found: {INPUT_CSV_PATH}"
data = pd.read_csv(INPUT_CSV_PATH)
assert {'source', 'close', 'far'}.issubset(data.columns), \
    f"expected (source, close, far) columns, got {list(data.columns)}"
print(f"Loaded: {len(data):,} triplets across {data['source'].nunique()} sources")

if FRACTION < 1.0:
    before = len(data)
    data = (data.groupby('source', group_keys=False)
                .sample(frac=FRACTION, random_state=SEED)
                .reset_index(drop=True))
    print(f"Per-source {FRACTION:.0%} sample: {before:,} -> {len(data):,} triplets")

per_source = data.groupby('source').size()
print(f"Pairs/source: min={per_source.min()}, max={per_source.max()}, "
      f"mean={per_source.mean():.0f}, median={int(per_source.median())}")
if 'group' in data.columns:
    print("\nLogical group distribution:")
    for grp, count in data['group'].value_counts().items():
        print(f"  {grp:30s} {count:>10,}  ({100*count/len(data):.1f}%)")


# **Section 2: SWAG + Adversarial Filtering**

Train shallow stylistic classifiers on the raw benchmark, then iteratively remove the easiest pairs until the ensemble drops to near-chance. Output: a filtered benchmark CSV saved to Drive at `OUTPUT_CSV_PATH`.

Sub-sections:
- 2.1 SWAG diagnostic models (tokenization, vocab, GloVe, models, training loop)
- 2.2 Adversarial filtering pipeline + cleanup


**1. Setup & Imports, Load Data**

In [ ]:
# Section 2.1 — SWAG-specific imports & config
# (general imports, Drive mount, and `data` loading happen in Section 0)

from sklearn.model_selection import StratifiedKFold, KFold
import spacy
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner', 'lemmatizer'])

# (SWAG hyperparameters are defined in the Hyper Parameters cell at the top)

assert 'data' in globals(), "Run Section 0.3 first — `data` should be the expanded (source, close, far) DataFrame."
print(f"Using benchmark from Section 0.3: {len(data)} pairs across {data['source'].nunique()} sources")
print(f"Columns: {list(data.columns)}")


Using benchmark from Section 0.3: 1058046 pairs across 500 sources
Columns: ['source', 'close', 'far', 'triplet_id', 'image_id', 'original_close', 'original_far', 'anchor_id_close', 'anchor_id_far', 'enrichment_label_close', 'enrichment_label_far', 'rule_type_close', 'rule_type_far']


 **2. Prepare Text and Labels**

In [ ]:
##############################################
# Prepare texts — spaCy tokenization + POS
##############################################
# Dict-based: tokenize each unique text once, lookup by string.

unique_close = data['close'].unique().tolist()
unique_far = data['far'].unique().tolist()
all_unique_texts = list(set(unique_close + unique_far))

# --- Run spaCy on all unique texts (one pass → word tokens + POS tags) ---
print(f"Running spaCy on {len(all_unique_texts)} unique texts...")
tokens_map = {}     # text → list of lowercased token strings
pos_raw_map = {}    # text → list of (word_lower, pos_tag) tuples

for text, doc in zip(all_unique_texts, nlp.pipe(all_unique_texts, batch_size=512, n_process=SPACY_PROCS)):
    words = [t.text.lower() for t in doc]
    pos_pairs = [(t.text.lower(), t.pos_) for t in doc]
    tokens_map[text] = words
    pos_raw_map[text] = pos_pairs

print(f"Done. Tokenized {len(tokens_map)} unique texts.")

# Quick stats
close_tok_lens = [len(tokens_map[t]) for t in unique_close]
far_tok_lens = [len(tokens_map[t]) for t in unique_far]

print(f"\nClassification: {len(unique_close)} unique close vs {len(unique_far)} unique far")
print(f"Chance accuracy: 50.0%")
print(f"\nToken lengths (spaCy):")
print(f"  Close: mean={np.mean(close_tok_lens):.1f}, std={np.std(close_tok_lens):.1f}, range=[{min(close_tok_lens)}, {max(close_tok_lens)}]")
print(f"  Far:   mean={np.mean(far_tok_lens):.1f}, std={np.std(far_tok_lens):.1f}, range=[{min(far_tok_lens)}, {max(far_tok_lens)}]")

# Example
sample = unique_close[0]
print(f"\nExample:")
print(f"  Close[0]: {sample[:80]}...")
print(f"  Tokens:   {tokens_map[sample][:10]}")
print(f"  POS raw:  {pos_raw_map[sample][:10]}")

Running spaCy on 72223 unique texts...
Done. Tokenized 72223 unique texts.

Classification: 25837 unique close vs 46386 unique far
Chance accuracy: 50.0%

Token lengths (spaCy):
  Close: mean=42.8, std=7.1, range=[14, 82]
  Far:   mean=42.7, std=7.3, range=[14, 101]

Example:
  Close[0]: 5 men are sitting on the end of a sailing boat on a lake, and one man throws a h...
  Tokens:   ['5', 'men', 'are', 'sitting', 'on', 'the', 'end', 'of', 'a', 'sailing']
  POS raw:  [('5', 'NUM'), ('men', 'NOUN'), ('are', 'AUX'), ('sitting', 'VERB'), ('on', 'ADP'), ('the', 'DET'), ('end', 'NOUN'), ('of', 'ADP'), ('a', 'DET'), ('sailing', 'NOUN')]


**3. GloVe & Build Vocabularies**

In [ ]:
# --- Download GloVe ---
GLOVE_PATH = 'glove.6B.100d.txt'

if not os.path.exists(GLOVE_PATH):
    zip_path = 'glove.6B.zip'
    if not os.path.exists(zip_path):
        print("Downloading GloVe embeddings (862MB)...")
        urllib.request.urlretrieve(
            'https://nlp.stanford.edu/data/glove.6B.zip', zip_path
        )
    print("Extracting glove.6B.100d.txt...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extract('glove.6B.100d.txt')
    print("Done.")
else:
    print("GloVe already downloaded.")


# --- Word vocabulary (from spaCy tokens, lowercased) ---
# SWAG: SingleIdTokenIndexer(namespace='tokens', lowercase_tokens=True)
all_flat_tokens = [w for toks in tokens_map.values() for w in toks]

token_counts = Counter(all_flat_tokens)
vocab_list = ['<PAD>', '<UNK>', '@@bos@@', '@@eos@@'] + \
             [w for w, c in token_counts.most_common()]
word2idx = {w: i for i, w in enumerate(vocab_list)}
vocab_size = len(word2idx)
print(f"Word vocabulary size: {vocab_size}")


# --- POS vocabulary (SWAG sample_candidates.py lines 37-42, 199-201) ---
# SWAG logic: if word is in top-100 → keep it (lowercased). Else → spaCy .pos_ tag.
top_words = set(w for w, _ in token_counts.most_common(TOP_K_WORDS))
print(f"Top {TOP_K_WORDS} words (kept as-is): {sorted(top_words)[:20]}...")

# spaCy Universal POS tags (SWAG line 38: spacy.parts_of_speech.NAMES)
spacy_pos_tags = set()
for name in spacy.parts_of_speech.NAMES.values():
    if len(name) > 0:
        spacy_pos_tags.add(name)
print(f"spaCy POS tags: {sorted(spacy_pos_tags)}")

# Build POS sequences using exact SWAG logic
pos_map = {}
for text, pos_pairs in pos_raw_map.items():
    pos_seq = []
    for word_lower, pos_tag in pos_pairs:
        if word_lower in top_words:
            pos_seq.append(word_lower)   # keep common word
        else:
            pos_seq.append(pos_tag)      # replace with POS tag
    pos_map[text] = pos_seq

print(f"\n✓ POS sequences built for {len(pos_map)} texts.")

# Build POS vocab
all_pos_flat = [p for seq in pos_map.values() for p in seq]

pos_counts = Counter(all_pos_flat)
pos_vocab_list = ['<PAD>', '<UNK>', '@@bos@@', '@@eos@@'] + \
                 [w for w, c in pos_counts.most_common()]
pos2idx = {w: i for i, w in enumerate(pos_vocab_list)}
pos_vocab_size = len(pos2idx)
print(f"POS vocabulary size: {pos_vocab_size}")

# Show example
sample = unique_close[0]
print(f"\nExample:")
print(f"  Words: {tokens_map[sample][:12]}")
print(f"  POS:   {pos_map[sample][:12]}")


# --- Load GloVe into embedding matrix ---
print("\nLoading GloVe embeddings...")
glove = {}
with open(GLOVE_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.split()
        word = parts[0]
        if word in word2idx:
            glove[word] = np.array(parts[1:], dtype=np.float32)

embedding_matrix = np.random.randn(vocab_size, GLOVE_DIM).astype(np.float32) * 0.01
embedding_matrix[0] = 0  # PAD = zeros
for word, idx in word2idx.items():
    if word in glove:
        embedding_matrix[idx] = glove[word]

coverage = sum(1 for w in word2idx if w in glove) / len(word2idx)
print(f"GloVe coverage: {coverage:.1%} of vocabulary")

 **4.DataSet and DataLoader**

In [ ]:
##############################################
# Dataset & DataLoader — dict-based, with option randomization
##############################################

def compute_text_feats(words, text):
    """Compute 8 statistical features for a standalone sentence."""
    return [
        len(words),                                                    # token count
        len(text),                                                     # char count
        np.mean([len(w) for w in words]) if words else 0,              # avg word length
        np.max([len(w) for w in words]) if words else 0,               # max word length
        len(set(words)) / max(len(words), 1),                          # lexical diversity
        text.count(',') + text.count('.') + text.count('!') + text.count('?'),  # punctuation count
        sum(1 for w in words if w in top_words) / max(len(words), 1),  # common word ratio
        len(words) - len(set(words)),                                  # repeated word count
    ]


class SwagDataset(Dataset):
    """
    Same-source pairing (Section 2):
      Each row in data is one (close, far) pair.
    Option order randomized: 50% close first, 50% far first.
    Uses tokens_map / pos_map dicts for lookup.
    """

    def __init__(self, row_indices, data_df, seed=42):
        self.row_indices = row_indices
        self.data = data_df
        rng = np.random.RandomState(seed)
        self.swap = rng.randint(0, 2, size=len(row_indices))

    def __len__(self):
        return len(self.row_indices)

    def _encode(self, text):
        words = tokens_map[text]
        pos_seq = pos_map[text]
        UNK = word2idx['<UNK>']
        POS_UNK = pos2idx['<UNK>']

        word_ids = ([word2idx['@@bos@@']] +
                    [word2idx.get(w, UNK) for w in words] +
                    [word2idx['@@eos@@']])
        pos_ids = ([pos2idx['@@bos@@']] +
                   [pos2idx.get(p, POS_UNK) for p in pos_seq] +
                   [pos2idx['@@eos@@']])

        feats = compute_text_feats(words, text)
        return (np.array(word_ids, dtype=np.int64),
                np.array(pos_ids, dtype=np.int64),
                np.array(feats, dtype=np.float32))

    def __getitem__(self, idx):
        row = self.data.iloc[self.row_indices[idx]]
        close_w, close_p, close_f = self._encode(row['close'])
        far_w, far_p, far_f = self._encode(row['far'])

        if self.swap[idx] == 0:
            label = 0  # close is option 0
            o0_w, o1_w = close_w, far_w
            o0_p, o1_p = close_p, far_p
            o0_f, o1_f = close_f, far_f
        else:
            label = 1  # close is option 1
            o0_w, o1_w = far_w, close_w
            o0_p, o1_p = far_p, close_p
            o0_f, o1_f = far_f, close_f

        return {
            'o0_word_ids': o0_w, 'o1_word_ids': o1_w,
            'o0_pos_ids': o0_p, 'o1_pos_ids': o1_p,
            'lm_feats': np.stack([o0_f, o1_f]),
            'label': label,
        }


def collate_fn(batch):
    """Pad and stack into [B, 2, L]."""
    lm_feats = torch.FloatTensor(np.stack([b['lm_feats'] for b in batch]))
    labels = torch.LongTensor([b['label'] for b in batch])

    max_len = max(
        max(len(b['o0_word_ids']), len(b['o1_word_ids']))
        for b in batch
    )

    def pad(arr, length):
        return np.pad(arr, (0, length - len(arr)), constant_values=0)

    word_ids = torch.LongTensor(np.stack([
        np.stack([pad(b['o0_word_ids'], max_len),
                  pad(b['o1_word_ids'], max_len)])
        for b in batch
    ]))

    pos_ids = torch.LongTensor(np.stack([
        np.stack([pad(b['o0_pos_ids'], max_len),
                  pad(b['o1_pos_ids'], max_len)])
        for b in batch
    ]))

    return {
        'lm_feats': lm_feats,           # [B, 2, 8]
        'ending_word_ids': word_ids,     # [B, 2, L]
        'postags_word_ids': pos_ids,     # [B, 2, L]
        'labels': labels,                # [B]
    }


# Quick test
test_ds = SwagDataset(np.arange(min(4, len(data))), data, seed=0)
test_loader = DataLoader(test_ds, batch_size=2, collate_fn=collate_fn)
test_batch = next(iter(test_loader))
print("Batch shapes:")
for k, v in test_batch.items():
    print(f"  {k}: {v.shape}")

Batch shapes:
  lm_feats: torch.Size([2, 2, 8])
  ending_word_ids: torch.Size([2, 2, 46])
  postags_word_ids: torch.Size([2, 2, 46])
  labels: torch.Size([2])


 **5. Model Definitions**

In [ ]:
##############################################
# Models — adapted from SWAG classifiers.py
# Removed ctx_indicator / binary_feature_embedding
# since our texts are independent sentences (no shared prefix)
##############################################

def reshape(f):
    """SWAG decorator: [batch, num_options, ...] → [batch*num_options, ...] → back."""
    def wrapper(self, *args, **kwargs):
        sizes = [x.size() for x in args]
        batch_size, num_ex = sizes[0][:2]
        flat_args = [x.reshape((-1,) + x.size()[2:]) for x in args]
        res = f(self, *flat_args, **kwargs)
        if isinstance(res, tuple):
            return tuple([x.reshape((batch_size, num_ex,) + x.size()[1:]) for x in res])
        return res.reshape((batch_size, num_ex,) + res.size()[1:])
    return wrapper


class LMFeatsModel(nn.Module):
    """MLP on 8 statistical features. (SWAG classifiers.py lines 37-64)"""
    def __init__(self, input_dim=8, hidden_dim=1024):
        super().__init__()
        self.mapping = nn.Sequential(
            nn.Linear(input_dim, hidden_dim, bias=True),
            nn.SELU(),
            nn.AlphaDropout(p=0.2),
        )
        self.prediction = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim, bias=True),
            nn.SELU(),
            nn.AlphaDropout(p=0.2),
            nn.Linear(hidden_dim, 1, bias=False),
        )

    @reshape
    def forward(self, feats):
        inter_feats = self.mapping(feats)
        preds = self.prediction(inter_feats).squeeze(1)
        return preds, inter_feats


class BoWModel(nn.Module):
    """Averaged GloVe → Linear. (SWAG classifiers.py lines 106-141)"""
    def __init__(self, pretrained_weights, embed_dim=100):
        super().__init__()
        self.embeds = nn.Embedding.from_pretrained(
            torch.FloatTensor(pretrained_weights), freeze=False, padding_idx=0
        )
        self.embed_dim = embed_dim
        self.embedding_to_label = nn.Linear(self.embed_dim, 1, bias=False)

    @reshape
    def forward(self, word_ids):
        embeds = self.embeds(word_ids)
        mask = (word_ids != 0).long()
        seq_lengths = mask.sum(-1, keepdim=True).float()
        seq_lengths[seq_lengths < 1] = 1.0
        inter_feats = embeds.sum(1) / seq_lengths
        preds = self.embedding_to_label(inter_feats).squeeze(1)
        return preds, inter_feats


class CNNModel(nn.Module):
    """Conv1D over GloVe embeddings. (SWAG classifiers.py lines 144-182, without ctx_indicator)"""
    def __init__(self, pretrained_weights, embed_dim=100,
                 window_sizes=(2, 3, 4, 5), num_filters=128):
        super().__init__()
        self.embeds = nn.Embedding.from_pretrained(
            torch.FloatTensor(pretrained_weights), freeze=False, padding_idx=0
        )
        # No binary_feature_embedding — no shared prefix to mark
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=ws, padding=ws - 1)
            for ws in window_sizes
        ])
        self.fc = nn.Linear(num_filters * len(window_sizes), 1, bias=False)

    @reshape
    def forward(self, word_ids):
        embeds = self.embeds(word_ids)                 # [B, L, 100]
        embeds_t = embeds.transpose(1, 2)              # [B, 100, L]
        conv_reps = [F.relu(conv(embeds_t)).max(2)[0] for conv in self.convs]
        inter_feats = torch.cat(conv_reps, 1)          # [B, 512]
        preds = self.fc(inter_feats).squeeze(1)
        return preds, inter_feats


class BLSTMModel(nn.Module):
    """BiLSTM over POS tags. (SWAG classifiers.py lines 185-233, without ctx_indicator)"""
    def __init__(self, pos_vocab_size, embed_dim=100, hidden_size=200,
                 dropout=0.3, maxpool=True):
        super().__init__()
        self.embeds = nn.Embedding(pos_vocab_size, embed_dim, padding_idx=0)
        # No binary_feature_embedding
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        self.maxpool = maxpool
        self.fc = nn.Linear(hidden_size * 2, 1, bias=False)

    @reshape
    def forward(self, word_ids):
        embeds = self.embeds(word_ids)                 # [B, L, 100]
        mask = (word_ids != 0).long()
        lengths = mask.sum(1).cpu().clamp(min=1)

        packed = nn.utils.rnn.pack_padded_sequence(
            embeds, lengths, batch_first=True, enforce_sorted=False
        )
        output, _ = self.lstm(packed)
        output, _ = nn.utils.rnn.pad_packed_sequence(output, batch_first=True)

        if self.maxpool:
            mask_expanded = mask.unsqueeze(2).float()
            output = output * mask_expanded + (1 - mask_expanded) * (-1e9)
            reps = output.max(1)[0]
        else:
            hs = output.size(2) // 2
            reps = torch.cat((output[torch.arange(output.size(0)), lengths-1, :hs],
                              output[:, 0, hs:]), 1)
        return self.fc(reps).squeeze(1), reps


class Ensemble(nn.Module):
    """
    All sub-models → concat → MLP. (SWAG classifiers.py lines 236-273)
    Concat: BoW(100) + MLP(1024) + LSTM(400) + CNN(512) = 2036
    No ctx_indicator passed to any sub-model.
    """
    def __init__(self, pretrained_weights, pos_vocab_size):
        super().__init__()
        self.fasttext_model = BoWModel(pretrained_weights, embed_dim=100)
        self.mlp_model = LMFeatsModel(input_dim=8, hidden_dim=1024)
        self.lstm_pos_model = BLSTMModel(pos_vocab_size, maxpool=True)
        self.cnn_model = CNNModel(pretrained_weights)

        self.mlp = nn.Sequential(
            nn.Linear(100 + 1024 + 400 + 4 * 128, 2048, bias=True),
            nn.SELU(),
            nn.AlphaDropout(p=0.2),
            nn.Linear(2048, 1, bias=False),
        )

    def forward(self, lm_feats, ending_word_ids, postags_word_ids):
        results = {}
        results['mlp'], mlp_feats = self.mlp_model(lm_feats)
        results['fasttext'], fasttext_feats = self.fasttext_model(ending_word_ids)
        results['cnn'], cnn_feats = self.cnn_model(ending_word_ids)
        results['lstm_pos'], lstm_feats = self.lstm_pos_model(postags_word_ids)
        results['ensemble'] = self.mlp(
            torch.cat((mlp_feats, fasttext_feats, cnn_feats, lstm_feats), 2)
        ).squeeze(2)
        return results

    def predict(self, lm_feats, ending_word_ids, postags_word_ids):
        results = self.forward(lm_feats, ending_word_ids, postags_word_ids)
        return {k: F.softmax(v, 1).data.cpu().numpy() for k, v in results.items()}

**6. Training. Loop**

In [ ]:
def train_and_evaluate(model, train_loader, val_loader, num_epochs=NUM_EPOCHS):
    """
    Training loop from Ensemble.fit() (classifiers.py lines 305-354).
    Label-aware: uses actual labels, not hardcoded zeros.
    """
    optimizer = optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        weight_decay=1e-6, lr=LR
    )

    best_val_accs = None
    best_val_ensemble = 0.0
    patience = 0
    PATIENCE = 3

    for epoch in range(num_epochs):
        # --- Train ---
        model.train()
        train_losses = []
        train_accs = {k: [] for k in ['mlp', 'fasttext', 'cnn', 'lstm_pos', 'ensemble']}

        for batch in train_loader:
            labels = batch.pop('labels').to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            results = model(**batch)

            losses = {
                f'{k}-loss': F.cross_entropy(v, labels)
                for k, v in results.items()
            }
            loss = sum(losses.values())

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_losses.append(loss.item())
            for k, v in results.items():
                train_accs[k].append((v.argmax(1) == labels).float().mean().item())

        # --- Validate ---
        model.eval()
        val_accs = {k: [] for k in ['mlp', 'fasttext', 'cnn', 'lstm_pos', 'ensemble']}
        with torch.no_grad():
            for batch in val_loader:
                labels = batch.pop('labels').to(DEVICE)
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                results = model(**batch)
                for k, v in results.items():
                    val_accs[k].append((v.argmax(1) == labels).float().mean().item())

        val_accs_mean = {k: np.mean(v) for k, v in val_accs.items()}
        train_accs_mean = {k: np.mean(v) for k, v in train_accs.items()}

        print(f"  Epoch {epoch+1:2d}/{num_epochs}  |  "
              f"loss={np.mean(train_losses):.4f}  |  "
              f"train_ens={train_accs_mean['ensemble']:.3f}  |  "
              f"val: " +
              "  ".join(f"{k}={v:.3f}" for k, v in val_accs_mean.items()))

        # Early stopping with patience
        if val_accs_mean['ensemble'] > best_val_ensemble:
            best_val_ensemble = val_accs_mean['ensemble']
            best_val_accs = val_accs_mean.copy()
            patience = 0
        else:
            patience += 1
            if patience >= PATIENCE and epoch > 2:
                print(f"  -> Early stopping (no improvement for {PATIENCE} epochs)")
                return best_val_accs

    return val_accs_mean

In [ ]:
# ============================================================
# 1. Initialize Active Pairs (start with ALL pairs)
# ============================================================

def init_all_active(data):
    """
    Start with every (source, close, far) row as active.
    Returns dict: source → set of (close, far) tuples
    """
    active = {}
    for src in data['source'].unique():
        sub = data[data['source'] == src]
        pairs = set(zip(sub['close'], sub['far']))
        active[src] = pairs
    return active


def active_to_dataframe(active):
    rows = []
    for src, pairs in active.items():
        for close, far in sorted(pairs):  # <-- add sorted()
            rows.append({'source': src, 'close': close, 'far': far})
    return pd.DataFrame(rows).reset_index(drop=True)


def active_stats(active, initial_total):
    """Quick stats on current active state."""
    counts = {s: len(pairs) for s, pairs in active.items()}
    total = sum(counts.values())
    return {
        'total_active': total,
        'pct_remaining': total / initial_total,
        'per_source_min': min(counts.values()),
        'per_source_max': max(counts.values()),
        'per_source_mean': np.mean(list(counts.values())),
        'sources_with_zero': sum(1 for c in counts.values() if c == 0),
    }


# Initialize
active = init_all_active(data)
initial_counts = {s: len(pairs) for s, pairs in active.items()}
initial_total = sum(initial_counts.values())

print(f"Starting benchmark:")
print(f"  Sources: {len(active)}")
print(f"  Total pairs: {initial_total}")
print(f"  Pairs per source: min={min(initial_counts.values())}, "
      f"max={max(initial_counts.values())}, "
      f"mean={np.mean(list(initial_counts.values())):.1f}")

In [ ]:
# ============================================================
# 2. Score Pairs with Trained Ensemble
# ============================================================

def score_pairs(model, pairs_df, seed=0):
    """
    Run trained ensemble over pairs_df. Returns per-pair confidence
    that close > far (1.0 = fully confident correct, 0.0 = wrong).
    """
    if len(pairs_df) == 0:
        return np.array([])

    ds = SwagDataset(np.arange(len(pairs_df)), pairs_df, seed=seed)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn)
    model.eval()
    all_scores = []
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop('labels').to(DEVICE)
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            results = model(**batch)
            probs = F.softmax(results['ensemble'], dim=1)
            close_scores = probs.gather(1, labels.unsqueeze(1)).squeeze(1)
            all_scores.append(close_scores.cpu().numpy())

    return np.concatenate(all_scores)

In [ ]:
# ============================================================
# 3. One AF Iteration — remove easy pairs (above chance only)
# ============================================================

def af_iteration(active, iteration, seed, removal_rate=0.1,
                 easy_threshold=0.5, verbose=True):
    # --- Step 1: Source-level split ---
    sources = sorted(active.keys())
    rng = np.random.RandomState(seed + iteration)
    shuffled = sources.copy()
    rng.shuffle(shuffled)
    split_pt = int(len(shuffled) * 0.8)
    train_srcs = shuffled[:split_pt]
    val_srcs = shuffled[split_pt:]

    # --- Step 2: Build train/val DataFrames ---
    train_active = {s: active[s] for s in train_srcs}
    val_active = {s: active[s] for s in val_srcs}
    train_df = active_to_dataframe(train_active)
    val_df = active_to_dataframe(val_active)

    if verbose:
        total = sum(len(v) for v in active.values())
        print(f"\n{'='*60}")
        print(f"AF Iteration {iteration}")
        print(f"  Sources: train={len(train_srcs)}, val={len(val_srcs)}")
        print(f"  Active pairs: train={len(train_df)}, val={len(val_df)}, "
              f"total={total}")
        print(f"{'='*60}")

    # --- Optional cap on TRAINING pairs (big-benchmark speedup; scoring unaffected) ---
    if MAX_TRAIN_PAIRS_PER_ITER is not None and len(train_df) > MAX_TRAIN_PAIRS_PER_ITER:
        train_df = train_df.sample(n=MAX_TRAIN_PAIRS_PER_ITER,
                                   random_state=seed + iteration).reset_index(drop=True)
        if verbose:
            print(f"  Train capped to {len(train_df):,} pairs (MAX_TRAIN_PAIRS_PER_ITER)")

    # --- Step 3: Train ensemble from scratch ---
    train_ds = SwagDataset(np.arange(len(train_df)), train_df,
                           seed=seed + iteration)
    val_ds = SwagDataset(np.arange(len(val_df)), val_df,
                         seed=seed + iteration + 10000)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=collate_fn, drop_last=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=collate_fn, num_workers=2)

    model = Ensemble(pretrained_weights=embedding_matrix,
                     pos_vocab_size=pos_vocab_size).to(DEVICE)
    val_accs = train_and_evaluate(model, train_loader, val_loader)

    # --- Step 4: Score all active val pairs ---
    val_scores = score_pairs(model, val_df, seed=seed + iteration + 10000)

    # --- Step 5: Remove easy pairs (above chance only) ---
    updated_active = {s: set(v) for s, v in active.items()}
    total_removed = 0
    removal_details = {}

    row_cursor = 0
    for src in val_srcs:
        src_pairs = sorted(active[src])
        n_src = len(src_pairs)
        if n_src == 0:
            continue

        src_scores = val_scores[row_cursor:row_cursor + n_src]
        row_cursor += n_src

        # Easy = model confidently CORRECT (score > threshold)
        easy_list = [(pair, score) for pair, score in
                     zip(src_pairs, src_scores) if score > easy_threshold]

        if len(easy_list) == 0:
            removal_details[src] = 0
            continue

        easy_list.sort(key=lambda x: -x[1])
        n_remove = max(1, int(len(easy_list) * removal_rate))

        if len(updated_active[src]) - n_remove < 1:
            n_remove = max(0, len(updated_active[src]) - 1)

        to_remove = [pair for pair, _ in easy_list[:n_remove]]
        updated_active[src] -= set(to_remove)
        total_removed += n_remove
        removal_details[src] = n_remove

    if verbose:
        print(f"  Removed {total_removed} pairs from {len(val_srcs)} val sources")

    return updated_active, val_accs, total_removed, removal_details

In [ ]:
# ============================================================
# 3b. Checkpoint / resume helpers
# ============================================================
# AF is deterministic per iteration (RandomState(SEED + iteration), fresh model each time),
# so resuming at iteration t reproduces exactly the run that would have happened.
# State is stored compactly as ROW IDS into `data` (2.6M pairs ~ 21MB, not GBs of text).

import glob as _glob

def _resolve_ckpt_dir():
    """One subfolder per AF run. RUN_NAME pins a folder explicitly; otherwise:
    RESUME -> latest run whose state.json is not phase='done'; else a fresh run_<stamp>/."""
    os.makedirs(CKPT_ROOT, exist_ok=True)
    if RUN_NAME:
        d = f'{CKPT_ROOT}/{RUN_NAME}'
        os.makedirs(d, exist_ok=True)
        return d
    runs = sorted(p for p in _glob.glob(f'{CKPT_ROOT}/run_*') if os.path.isdir(p))
    if RESUME:
        for d in reversed(runs):                       # newest first
            sj = f'{d}/state.json'
            if os.path.exists(sj):
                with open(sj) as fh:
                    if json.load(fh).get('phase') != 'done':
                        return d                       # unfinished -> resume here
    d = f"{CKPT_ROOT}/run_{datetime.now().strftime('%Y%m%d_%H%M')}"
    os.makedirs(d, exist_ok=True)
    return d

CKPT_DIR = _resolve_ckpt_dir()

# Snapshot this run's config on first use (audit trail per run folder)
if not os.path.exists(f'{CKPT_DIR}/af_config.json'):
    with open(f'{CKPT_DIR}/af_config.json', 'w') as fh:
        json.dump({'input_csv': INPUT_CSV_PATH, 'output_csv': OUTPUT_CSV_PATH,
                   'fraction': FRACTION, 'n_data': len(data), 'seed': SEED,
                   'removal_rate': REMOVAL_RATE, 'easy_threshold': EASY_THRESHOLD,
                   'accuracy_target': ACCURACY_TARGET, 'min_retention': MIN_RETENTION,
                   'max_iters': MAX_ITERS, 'batch_size': BATCH_SIZE,
                   'max_train_pairs_per_iter': MAX_TRAIN_PAIRS_PER_ITER,
                   'cleanup': [CLEANUP_THRESHOLD, CLEANUP_REMOVAL_RATE, CLEANUP_MAX_ITERS,
                               CLEANUP_MAX_REMOVAL, CLEANUP_TARGET],
                   'created': datetime.now().isoformat()}, fh, indent=2)

# Stable pair <-> id mapping over the loaded benchmark
_PAIR_ID = {k: i for i, k in enumerate(zip(data['source'], data['close'], data['far']))}
assert len(_PAIR_ID) == len(data), "duplicate (source, close, far) rows in data"

# Content fingerprint of the loaded benchmark (order-sensitive). Stored in every
# checkpoint; resume hard-fails if the CSV was regenerated/reordered since the
# run started (the n_data count check alone cannot catch same-size content drift).
import hashlib
DATA_FPRINT = hashlib.md5(
    pd.util.hash_pandas_object(data[['source', 'close', 'far']], index=False)
      .values.tobytes()).hexdigest()

def active_to_ids(active):
    ids = [ _PAIR_ID[(s, c, f)] for s, pairs in active.items() for (c, f) in pairs ]
    return np.array(sorted(ids), dtype=np.int64)

def ids_to_active(ids):
    sub = data.iloc[ids]
    act = {s: set(zip(g['close'], g['far'])) for s, g in sub.groupby('source')}
    for s in data['source'].unique():
        act.setdefault(s, set())          # keep emptied sources as empty sets
    return act

def _atomic_np(path, arr):
    np.save(path + '.tmp.npy', arr); os.replace(path + '.tmp.npy', path)

def save_af_state(phase, main_iter=-1, cleanup_iter=-1, stop_reason='',
                  pre_cleanup_total=0, total_cleanup_removed=0):
    """Persist full AF state to CKPT_DIR. Reads active/best_active/history/cleanup_history
    from globals. Atomic-ish: state.json is written last."""
    a_ids, b_ids = active_to_ids(active), active_to_ids(best_active)
    assert len(a_ids) == sum(len(v) for v in active.values()) == len(np.unique(a_ids)), \
        "checkpoint invariant broken: active ids do not match active set 1:1"
    assert len(b_ids) == sum(len(v) for v in best_active.values()) == len(np.unique(b_ids)), \
        "checkpoint invariant broken: best ids do not match best set 1:1"
    _atomic_np(f'{CKPT_DIR}/active_ids.npy', a_ids)
    _atomic_np(f'{CKPT_DIR}/best_ids.npy',   b_ids)
    pd.DataFrame(history).to_csv(f'{CKPT_DIR}/history.csv', index=False)
    if 'cleanup_history' in globals():
        pd.DataFrame(cleanup_history).to_csv(f'{CKPT_DIR}/cleanup_history.csv', index=False)
    state = {'phase': phase, 'main_iter': int(main_iter), 'cleanup_iter': int(cleanup_iter),
             'best_acc': float(best_acc), 'best_iter': int(best_iter),
             'stop_reason': stop_reason, 'pre_cleanup_total': int(pre_cleanup_total),
             'total_cleanup_removed': int(total_cleanup_removed),
             'n_data': len(data), 'data_fprint': DATA_FPRINT,
             'saved_at': datetime.now().isoformat()}
    with open(f'{CKPT_DIR}/state.json.tmp', 'w') as fh:
        json.dump(state, fh, indent=2)
    os.replace(f'{CKPT_DIR}/state.json.tmp', f'{CKPT_DIR}/state.json')

def load_af_state():
    """Return the saved state dict (with id arrays + histories) or None."""
    if not os.path.exists(f'{CKPT_DIR}/state.json'):
        return None
    with open(f'{CKPT_DIR}/state.json') as fh:
        st = json.load(fh)
    assert st['n_data'] == len(data), (
        f"checkpoint was made on a benchmark of {st['n_data']} rows, but current data has "
        f"{len(data)} — same INPUT_CSV_PATH + FRACTION required for resume")
    if st.get('data_fprint') is not None:
        assert st['data_fprint'] == DATA_FPRINT, (
            "checkpoint content fingerprint does not match the loaded CSV — the benchmark "
            "file was regenerated or reordered since this run started; resume would be invalid")
    st['active_ids'] = np.load(f'{CKPT_DIR}/active_ids.npy')
    st['best_ids']   = np.load(f'{CKPT_DIR}/best_ids.npy')
    for k in ('active_ids', 'best_ids'):
        assert len(st[k]) == len(np.unique(st[k])) and (
            len(st[k]) == 0 or (st[k].min() >= 0 and st[k].max() < len(data))), \
            f"corrupt checkpoint: {k} has duplicates or out-of-range indices"
    st['history'] = (pd.read_csv(f'{CKPT_DIR}/history.csv').to_dict('records')
                     if os.path.getsize(f'{CKPT_DIR}/history.csv') > 1 else []) \
                    if os.path.exists(f'{CKPT_DIR}/history.csv') else []
    st['cleanup_history'] = (pd.read_csv(f'{CKPT_DIR}/cleanup_history.csv').to_dict('records')
                             if os.path.exists(f'{CKPT_DIR}/cleanup_history.csv')
                             and os.path.getsize(f'{CKPT_DIR}/cleanup_history.csv') > 1 else [])
    return st

# --- Session log: summary lines persist to the run folder; replayed on resume ---
LOG_PATH = f'{CKPT_DIR}/run_log.txt'

def log_print(msg=''):
    """print() that also appends to the run's log file (survives disconnects)."""
    print(msg)
    with open(LOG_PATH, 'a') as fh:
        fh.write(msg + '\n')

def replay_log():
    """Print all summary lines logged by previous sessions of this run."""
    if os.path.exists(LOG_PATH) and os.path.getsize(LOG_PATH) > 0:
        print('=' * 60)
        print('REPLAY - log from previous sessions of this run:')
        print('=' * 60)
        with open(LOG_PATH) as fh:
            print(fh.read(), end='')
        print('=' * 60 + '\nEND REPLAY\n' + '=' * 60)

# --- One-time self-test: active -> ids -> active must be the identity ---
_test_srcs = list(data['source'].unique()[:3])
_test_active = {s2: set(zip(data.loc[data['source'] == s2, 'close'],
                            data.loc[data['source'] == s2, 'far'])) for s2 in _test_srcs}
_rt = ids_to_active(active_to_ids(_test_active) if _test_active else np.array([], dtype=np.int64))
assert all(_rt.get(s2, set()) == _test_active[s2] for s2 in _test_srcs), \
    "checkpoint round-trip self-test FAILED — do not trust saved states"
del _test_srcs, _test_active, _rt
print("Checkpoint round-trip self-test: OK")

_st = load_af_state()
print(f"Run folder: {CKPT_DIR}")
print("Existing checkpoint: " + (f"phase={_st['phase']}, main_iter={_st['main_iter']}, "
      f"cleanup_iter={_st['cleanup_iter']}" if _st else "none (fresh start)"))

In [ ]:
# ============================================================
# 4. AF Pipeline — fixed config, dual stopping, resume-aware
# ============================================================
# (AF hyperparameters are defined in the Hyper Parameters cell at the top)

initial_counts = data.groupby('source').size().to_dict()
initial_total = int(sum(initial_counts.values()))

_ck = load_af_state() if RESUME else None
if _ck is not None:
    replay_log()   # show summary lines from previous sessions
    active      = ids_to_active(_ck['active_ids'])
    best_active = ids_to_active(_ck['best_ids'])
    best_acc, best_iter = _ck['best_acc'], _ck['best_iter']
    history     = _ck['history']
    stop_reason = _ck['stop_reason'] or 'max_iters'
    if _ck['phase'] == 'main':
        start_iter = _ck['main_iter'] + 1
        log_print(f"RESUMING main AF loop at iteration {start_iter} "
              f"(best={best_acc:.3f}@{best_iter}, {len(history)} iterations logged)")
    else:   # main_done / cleanup / done — main loop already finished
        start_iter = MAX_ITERS
        log_print(f"Main AF loop already finished (phase={_ck['phase']}) — skipping to cleanup/save.")
else:
    active = init_all_active(data)
    history = []
    best_acc = 1.0
    best_active = {s: set(v) for s, v in active.items()}
    best_iter = 0
    stop_reason = 'max_iters'
    start_iter = 0
    log_print(f"AF Pipeline starting fresh")

print(f"  Sources: {len(active)} | Total pairs: {initial_total}")
print(f"  Accuracy target: {ACCURACY_TARGET} | Min retention: {MIN_RETENTION:.0%}")
print(f"  Removal rate: {REMOVAL_RATE} | Easy threshold: {EASY_THRESHOLD}")
print()

for t in range(start_iter, MAX_ITERS):
    active, val_accs, n_removed, removal_details = af_iteration(
        active, iteration=t, seed=SEED,
        removal_rate=REMOVAL_RATE, easy_threshold=EASY_THRESHOLD,
        verbose=True,
    )

    ens_acc = val_accs['ensemble']
    total_active = sum(len(v) for v in active.values())
    pct_remaining = total_active / initial_total
    counts = [len(v) for v in active.values()]

    history.append({
        'iter': t, 'ensemble': ens_acc,
        'mlp': val_accs['mlp'], 'fasttext': val_accs['fasttext'],
        'cnn': val_accs['cnn'], 'lstm_pos': val_accs['lstm_pos'],
        'removed': n_removed, 'total_active': total_active,
        'pct_remaining': pct_remaining,
        'per_source_min': min(counts), 'per_source_mean': np.mean(counts),
        'sources_with_zero': sum(1 for c in counts if c == 0),
    })

    log_print(f"\n>> Iter {t}: ensemble={ens_acc:.3f} | "
          f"removed={n_removed} | "
          f"active={total_active}/{initial_total} ({pct_remaining:.1%}) | "
          f"per_src: min={min(counts)} mean={np.mean(counts):.1f}")

    if ens_acc < best_acc:
        best_acc = ens_acc
        best_active = {s: set(v) for s, v in active.items()}
        best_iter = t

    # --- checkpoint after the iteration (survives Colab disconnects) ---
    if t % CHECKPOINT_EVERY == 0:
        save_af_state(phase='main', main_iter=t, stop_reason='')
        print(f"   [ckpt] saved at iter {t}")

    if ens_acc <= ACCURACY_TARGET:
        stop_reason = f"accuracy_target ({ACCURACY_TARGET})"
        log_print(f"\n>> STOP: Ensemble accuracy {ens_acc:.3f} <= target")
        break

    if pct_remaining < MIN_RETENTION:
        stop_reason = f"min_retention ({MIN_RETENTION:.0%})"
        log_print(f"\n>> STOP: Benchmark at {pct_remaining:.1%} < minimum")
        break

save_af_state(phase='main_done', main_iter=len(history) - 1, stop_reason=stop_reason)

print(f"\n{'='*60}")
log_print(f"AF complete. Stop: {stop_reason}")
log_print(f"  Best ensemble: {best_acc:.3f} at iter {best_iter}")
best_total = sum(len(v) for v in best_active.values())
log_print(f"  Best benchmark: {best_total}/{initial_total} "
      f"({best_total/initial_total:.1%})")
print(f"{'='*60}")

# Use best snapshot going forward
active = best_active


REPLAY - log from previous sessions of this run:
RESUMING main AF loop at iteration 1 (best=0.713@0, 1 iterations logged)

>> Iter 1: ensemble=0.718 | removed=60679 | active=4245602/4366748 (97.2%) | per_src: min=5548 mean=8491.2

>> Iter 2: ensemble=0.726 | removed=60005 | active=4185597/4366748 (95.9%) | per_src: min=5548 mean=8371.2

>> Iter 3: ensemble=0.734 | removed=61409 | active=4124188/4366748 (94.4%) | per_src: min=5548 mean=8248.4

>> Iter 4: ensemble=0.714 | removed=58097 | active=4066091/4366748 (93.1%) | per_src: min=5548 mean=8132.2

>> Iter 5: ensemble=0.700 | removed=56177 | active=4009914/4366748 (91.8%) | per_src: min=5548 mean=8019.8

>> Iter 6: ensemble=0.713 | removed=55359 | active=3954555/4366748 (90.6%) | per_src: min=5548 mean=7909.1

>> Iter 7: ensemble=0.699 | removed=54091 | active=3900464/4366748 (89.3%) | per_src: min=5548 mean=7800.9

>> Iter 8: ensemble=0.703 | removed=54452 | active=3846012/4366748 (88.1%) | per_src: min=5548 mean=7692.0

>> Iter 9: en

In [ ]:
# Use best snapshot going forward into save (NO cleanup phase — removed by design)
active = {s: set(v) for s, v in best_active.items()}
print(f"Using best snapshot: {sum(len(v) for v in active.values()):,} pairs "
      f"(best ensemble {best_acc:.3f} at iter {best_iter})")


Using best snapshot: 1,082,135 pairs (best ensemble 0.545 at iter 108)


In [ ]:
# ============================================================
# 5. Save curated benchmark & logs
# ============================================================

from datetime import datetime
import matplotlib.pyplot as plt

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# --- Save curated benchmark (ALL input metadata columns re-attached) ---
# active_to_dataframe returns only (source, close, far); merge back everything else
# from `data` (triplet_id, image_id, original_*, anchor_id_*, labels, group, ...).
# Safe: (source, close, far) is unique in `data` (asserted when _PAIR_ID was built).
best_df = active_to_dataframe(best_active)
if len(data.columns) > 3:
    best_df = best_df.merge(data, on=['source', 'close', 'far'], how='left')
    assert len(best_df) == sum(len(v) for v in best_active.values()), \
        'metadata merge changed row count - (source, close, far) not unique in input'
best_df.to_csv(f'benchmark_af_{timestamp}.csv', index=False)
best_df.to_csv(OUTPUT_CSV_PATH, index=False)  # Drive copy — the notebook's product

print(f"Curated benchmark saved: {len(best_df)} pairs "
      f"({len(best_df)/initial_total:.1%} of original)")

# --- Per-source stats ---
src_counts = best_df.groupby('source').size()
print(f"\nPer-source distribution:")
print(f"  min={src_counts.min()}, max={src_counts.max()}, "
      f"mean={src_counts.mean():.1f}, median={src_counts.median():.0f}")

per_source_rows = []
for src in sorted(initial_counts.keys()):
    init_n = initial_counts[src]
    final_n = len(best_active.get(src, set()))
    per_source_rows.append({
        'source': src,
        'initial_pairs': init_n,
        'final_pairs': final_n,
        'removed': init_n - final_n,
        'pct_remaining': final_n / init_n if init_n > 0 else 0,
    })
per_source_df = pd.DataFrame(per_source_rows)
REPORT_DIR = os.path.dirname(OUTPUT_CSV_PATH)   # Drive — survives the runtime
per_source_df.to_csv(f'{REPORT_DIR}/af_per_source_{timestamp}.csv', index=False)

print(f"\nSources that lost >50% pairs: "
      f"{sum(1 for r in per_source_rows if r['pct_remaining'] < 0.5)}")
print(f"Sources that kept >80% pairs: "
      f"{sum(1 for r in per_source_rows if r['pct_remaining'] > 0.8)}")
print(f"Sources with zero pairs: "
      f"{sum(1 for r in per_source_rows if r['final_pairs'] == 0)}")

# --- Retention by metadata dimension (category / anchor / rule, close & far sides) ---
RETENTION_DIMS = ['enrichment_label_close', 'enrichment_label_far',
                  'rule_type_close', 'rule_type_far',
                  'anchor_id_close', 'anchor_id_far']
ret_rows = []
for col in RETENTION_DIMS:
    if col not in best_df.columns:
        continue
    init = data[col].value_counts()
    fin  = best_df[col].value_counts().reindex(init.index).fillna(0).astype(int)
    print(f"\nRetention by {col}:")
    for v in init.sort_values(ascending=False).index:
        pct = fin[v] / init[v] if init[v] else 0.0
        ret_rows.append({'dimension': col, 'value': v, 'initial_pairs': int(init[v]),
                         'final_pairs': int(fin[v]), 'pct_remaining': pct})
        print(f"  {str(v):30s} {init[v]:>10,} -> {fin[v]:>10,}  ({pct:.1%})")
if ret_rows:
    pd.DataFrame(ret_rows).to_csv(f'{REPORT_DIR}/af_retention_by_dimension_{timestamp}.csv',
                                  index=False)

# --- Survival grid: close category x far category (pct remaining per cell) ---
if {'enrichment_label_close', 'enrichment_label_far'}.issubset(best_df.columns):
    g_init = data.pivot_table(index='enrichment_label_close',
                              columns='enrichment_label_far', aggfunc='size', fill_value=0)
    g_fin  = best_df.pivot_table(index='enrichment_label_close',
                                 columns='enrichment_label_far', aggfunc='size', fill_value=0)
    grid = (g_fin.reindex_like(g_init).fillna(0) / g_init.replace(0, np.nan)).round(3)
    grid.to_csv(f'{REPORT_DIR}/af_retention_grid_{timestamp}.csv')
    print("\nSurvival grid (close label x far label, pct remaining):")
    print(grid.to_string())

# --- Artifact sanity: did AF change surface statistics? ---
print("\nSurface statistics (the artifact axes AF is supposed to neutralize):")
for name, df_ in (('pre-AF ', data), ('post-AF', best_df)):
    cl, fl = df_['close'].str.len(), df_['far'].str.len()
    print(f"  {name}: close_len={cl.mean():.1f}  far_len={fl.mean():.1f}  "
          f"gap={cl.mean()-fl.mean():+.1f} | "
          f"unique close={df_['close'].nunique():,}  unique far={df_['far'].nunique():,}")

# --- Anchor coverage: sources x far-anchors still represented ---
if 'anchor_id_far' in best_df.columns:
    cov_init = data.groupby('source')['anchor_id_far'].nunique()
    cov_fin  = best_df.groupby('source')['anchor_id_far'].nunique()
    lost = int((cov_fin.reindex(cov_init.index).fillna(0) < cov_init).sum())
    print(f"\nFar-anchor coverage: {lost}/{len(cov_init)} sources lost at least one "
          f"far anchor entirely (min surviving anchors per source: "
          f"{int(cov_fin.min()) if len(cov_fin) else 0})")

# --- Per-category retention (spec: per-category percentage remaining) ---
if 'group' in data.columns:
    init_grp  = data['group'].value_counts()
    final_grp = best_df['group'].value_counts()
    per_cat = pd.DataFrame({
        'initial_pairs': init_grp,
        'final_pairs':   final_grp.reindex(init_grp.index).fillna(0).astype(int),
    })
    per_cat['removed'] = per_cat['initial_pairs'] - per_cat['final_pairs']
    per_cat['pct_remaining'] = per_cat['final_pairs'] / per_cat['initial_pairs']
    per_cat = per_cat.sort_values('pct_remaining')
    print(f"\nPer-category retention:")
    for grp, r in per_cat.iterrows():
        print(f"  {grp:30s} {int(r['final_pairs']):>8,} / {int(r['initial_pairs']):>8,}"
              f"  ({r['pct_remaining']:.1%} remaining)")
    per_cat.to_csv(f'af_per_category_{timestamp}.csv')
    print(f"Per-category retention saved: af_per_category_{timestamp}.csv")

# --- Save iteration history ---
history_df = pd.DataFrame(history)
history_df.to_csv(f'af_history_{timestamp}.csv', index=False)
print(f"\nIteration history saved: {len(history_df)} iterations")
# --- Mark the run finished (previously done by the removed cleanup phase) ---
save_af_state(phase='done', main_iter=best_iter, stop_reason=stop_reason)
log_print("[ckpt] phase=done — AF finished, benchmark saved")


Curated benchmark saved: 1082135 pairs (24.8% of original)

Per-source distribution:
  min=430, max=4709, mean=2164.3, median=2016

Sources that lost >50% pairs: 496
Sources that kept >80% pairs: 0
Sources with zero pairs: 0

Retention by enrichment_label_close:
  freestyle                         827,506 ->    195,332  (23.6%)
  minor                             777,326 ->    185,349  (23.8%)
  sizes                             765,225 ->    185,415  (24.2%)
  visualizable                      724,513 ->    194,415  (26.8%)
  colors                            724,044 ->    184,636  (25.5%)
  texts                             548,134 ->    136,988  (25.0%)

Retention by enrichment_label_far:
  freestyle                         839,272 ->    204,284  (24.3%)
  minor                             769,554 ->    189,375  (24.6%)
  sizes                             765,071 ->    188,402  (24.6%)
  colors                            723,968 ->    180,859  (25.0%)
  visualizable                 

In [ ]:
# ============================================================
# 6. Convergence plots (main AF)
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
h = pd.DataFrame(history)

# --- Left: Accuracy convergence ---
ax = axes[0]
colors = {'mlp': '#1f77b4', 'fasttext': '#ff7f0e', 'cnn': '#2ca02c',
          'lstm_pos': '#d62728', 'ensemble': '#9467bd'}
for model_name in ['mlp', 'fasttext', 'cnn', 'lstm_pos', 'ensemble']:
    ax.plot(h['iter'], h[model_name], marker='.', label=model_name,
            color=colors[model_name])
ax.axhline(y=ACCURACY_TARGET, color='green', linestyle='--', alpha=0.7,
           label=f'target ({ACCURACY_TARGET})')
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='chance')
if best_iter in h['iter'].values:
    ax.axvline(x=best_iter, color='blue', linestyle=':', alpha=0.3,
               label=f'best iter ({best_iter})')
ax.set_xlabel('Iteration')
ax.set_ylabel('Val Accuracy')
ax.set_title('Accuracy: Main AF')
ax.legend(fontsize=6, ncol=2)
ax.grid(True, alpha=0.3)

# --- Middle: Retention ---
ax = axes[1]
ax.plot(h['iter'], h['pct_remaining'], marker='o', color='teal')
ax.axhline(y=MIN_RETENTION, color='orange', linestyle='--', alpha=0.7,
           label=f'min retention ({MIN_RETENTION:.0%})')
ax.set_xlabel('Iteration')
ax.set_ylabel('% Benchmark Remaining')
ax.set_title('Benchmark Retention')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# --- Right: Pairs removed per iteration ---
ax = axes[2]
ax.bar(h['iter'], h['removed'], color='coral', alpha=0.8)
ax.set_xlabel('Iteration')
ax.set_ylabel('Pairs Removed')
ax.set_title('Removals Per Iteration')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/af_pipeline_{timestamp}.png', dpi=100)
plt.show()

print(f"\nSummary: {stop_reason}")
print(f"  Main AF: {h['ensemble'].iloc[0]:.3f} -> {best_acc:.3f} (best @ iter {best_iter})")
final_total = sum(len(v) for v in active.values())
print(f"  Retention: 100% -> {final_total/initial_total:.1%}")
